In [1]:
from typing import List, Tuple, Dict
import csv
import os
from collections import defaultdict

In [2]:
def merge_popstat_and_genres(
        popstat: List[Tuple[str, float]],
        genres: List[Tuple[str, List[str]]],
        output_filepath: str,
        save_to_file: bool,
        ) -> List[Tuple[str, float, float, List[str]]]:
    assert (len (popstat) == len (genres)), "Number of elements in popstat and genres lists are not equal"

    # movie, popstat, unit value, genres
    merged_result: List[Tuple[str, float, float, List[str]]] = []

    popstat_sum = sum([popstat_val for _, popstat_val in popstat])

    for movie, popstat_val in popstat:
        appended = False
        for genre in genres:
            # if (movie.lower() == "праздник св. иоргена"):
            #     print(movie.lower(), "matching to: ", genre[0], genre[0].lower())
            if movie.lower() == genre[0].lower():
                unit_value = popstat_val / popstat_sum
                merged_result.append((movie, popstat_val, unit_value, genre[1]))
                appended = True
                break

        if not appended:
            raise ValueError(f"Genres for movie {movie} not found")

    if save_to_file:
        with open(output_filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)

            writer.writerow(['Movie', 'Popstat', 'UnitPopstat', 'Genres'])

            merged_result = sorted(merged_result, key=lambda x: x[1], reverse=True)
            for movie, popstat_val, unit_value, genre_list in merged_result:
                writer.writerow([movie, popstat_val, unit_value, ','.join(genre_list)])

    return merged_result

In [3]:
# Filepaths with CSV data
filepaths = [
    {
        "name": "metropol",
        "genge": "analysis/genres/metropol.csv",
        "popstat": "analysis/popstat/metropol.csv",
        "output": "analysis/merged/metropol.csv"
    },
    {
        "name": "udarnik",
        "genge": "analysis/genres/udarnik.csv",
        "popstat": "analysis/popstat/udarnik.csv",
        "output": "analysis/merged/udarnik.csv"
    },
    {
        "name": "orion",
        "genge": "analysis/genres/orion.csv",
        "popstat": "analysis/popstat/orion.csv",
        "output": "analysis/merged/orion.csv"
    },
]

In [4]:
# Merge popstat data with genres
def merge(filepaths: List[dict], save_to_file=False) -> Dict[str, List[Tuple[str, float, float, List[str]]]]:
    result = defaultdict(str)

    for paths in filepaths:
        genre_filepath = paths["genge"]
        popstat_filepath = paths["popstat"]

        genres = None
        popstat = None

        with open(genre_filepath, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Skip header row
            genres = [(row[0].strip(), [item.strip() for item in row[1].split(',')]) for row in reader]

        with open(popstat_filepath, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Skip header row
            popstat = [(row[0].strip(), float(row[1])) for row in reader]

        output_filepath = paths["output"]

        # Create directories if they do not exist
        os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

        print(output_filepath)
        merged_data = merge_popstat_and_genres(popstat, genres, output_filepath, save_to_file=save_to_file)

        result[paths["name"]] = merged_data

        print(f"Successfully merged {genre_filepath} and {popstat_filepath} into {output_filepath}")

    return result

data = merge(filepaths, save_to_file=True)

analysis/merged/metropol.csv
Successfully merged analysis/genres/metropol.csv and analysis/popstat/metropol.csv into analysis/merged/metropol.csv
analysis/merged/udarnik.csv
Successfully merged analysis/genres/udarnik.csv and analysis/popstat/udarnik.csv into analysis/merged/udarnik.csv
analysis/merged/orion.csv
Successfully merged analysis/genres/orion.csv and analysis/popstat/orion.csv into analysis/merged/orion.csv


In [5]:
def build_genre_distribution(filepath: str, save_to_file=False):
    with open(filepath, 'w', newline='', encoding='utf-8') as file:
        for cinema in data.keys():
            # print(cinema)
            file.write(f"Cinema: {cinema}\n")

            genre_weights = defaultdict(float)
            total_weight: float = 0.

            genre_movies = defaultdict(list)

            for movie, _popstat, unit_popstat, genre_list in data[cinema]:
                for genre in genre_list:
                    genre_weights[genre] += unit_popstat
                    total_weight += unit_popstat

                    genre_movies[genre].append((movie, unit_popstat))

            # normalize weights
            for genre, weight in genre_weights.items():
                genre_weights[genre] = weight / total_weight

            # print(genre_weights)
            # print(total_weight)

            sorted_items = sorted(genre_weights.items(), key=lambda x: x[1], reverse=True)
            for genre, weight in sorted_items:
                # print(genre, weight)
                file.write(f"{genre}: {round(weight * 100, 1)} ({weight})\n")

                sorted_movies = sorted(genre_movies[genre], key=lambda x: x[1], reverse=True)
                for movie, unit_popstat in sorted_movies:
                    # print(f"\t{movie}: {unit_popstat}")
                    file.write(f"\t{movie}: {round(unit_popstat * 100, 1)} ({unit_popstat})\n")
            # print()
            file.write("\n\n")

build_genre_distribution("analysis/genre_distribution.txt", save_to_file=True)